In [ ]:
!pip install -q llama-index-llms-Groq pymupdf
!pip install -q llama-index-embeddings-huggingface
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does 

In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import nest_asyncio

nest_asyncio.apply()

from google.colab import userdata
from llama_index.llms.groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

!mkdir -p sample_docs

In [ ]:
from google.colab import files
import os

def upload_pdf():
    """Upload a PDF file and return its path."""
    print("Please select a PDF file to upload:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.pdf'):
            pdf_path = os.path.join("sample_docs", filename)

            os.makedirs("sample_docs", exist_ok=True)

            with open(pdf_path, 'wb') as f:
                f.write(uploaded[filename])

            print(f"PDF saved to {pdf_path}")
            return pdf_path
        else:
            print(f"File {filename} is not a PDF. Please upload a PDF file.")

    return None

In [ ]:
pdf_path = upload_pdf()

Please select a PDF file to upload:


Saving sample-sdf-document.pdf to sample-sdf-document.pdf
PDF saved to sample_docs/sample-sdf-document.pdf


In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file using PyMuPDF."""
    doc = fitz.open(pdf_path)

    text = "\n".join([page.get_text() for page in doc])

    print(f"PDF: {pdf_path}")
    print(f"Number of pages: {len(doc)}")
    print(f"Extracted {len(text.split())} words from the PDF.")

    doc.close()

    return text

In [ ]:
if pdf_path:
    text = extract_text_from_pdf(pdf_path)
    print(text[:500])

PDF: sample_docs/sample-sdf-document.pdf
Number of pages: 3
Extracted 617 words from the PDF.
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could le


In [ ]:
from llama_index.core import Document
from typing import List

def load_pdf_with_pymupdf(pdf_path: str) -> List[Document]:
    """Load a PDF and convert it to LlamaIndex Document format using PyMuPDF."""
    doc = fitz.open(pdf_path)

    documents = []

    for i, page in enumerate(doc):
        text = page.get_text()

        if not text.strip():
            continue

        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": os.path.basename(pdf_path),
                    "page_number": i + 1,
                    "total_pages": len(doc)
                }
            )
        )

    doc.close()

    print(f"Processed {pdf_path}:")
    print(f"Extracted {len(documents)} pages with content")

    return documents

In [ ]:
pdf_docs = load_pdf_with_pymupdf(pdf_path)

if pdf_docs:
    print(pdf_docs[0].text[:500])
    print(pdf_docs[0].metadata)

Processed sample_docs/sample-sdf-document.pdf:
Extracted 3 pages with content
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could le
{'file_name': 'sample-sdf-document.pdf', 'page_number': 1, 'total_pages': 3}


In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter_overlap = SentenceSplitter(chunk_size=200, chunk_overlap=20)
chunks_overlap = splitter_overlap.get_nodes_from_documents(pdf_docs)
print(f"Total Overlapping Chunks Created: {len(chunks_overlap)}")

print(f"\nOverlap Example:")
print(f"Chunk 1: {chunks_overlap[0].text}")
print(f"Chunk 2: {chunks_overlap[1].text}")

Total Overlapping Chunks Created: 9

Overlap Example:
Chunk 1: Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could lead to 
brittleness or cracking of the plastic connectors.
Chunk 2: However, the operating temperature of ÄKTA ready flow kits is +2 C to 
+40 C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified,

In [ ]:

from llama_index.core import Settings
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Initialize Gemini LLM
llm = Groq(
    model="qwen/qwen3.6-27b",
)

Settings.llm = llm

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

def process_and_index_pdf(chunks):

    # Create vector index
    vector_index = VectorStoreIndex(chunks)
    print("Vector store created successfully!")
    return vector_index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
index = process_and_index_pdf(chunks_overlap)

Vector store created successfully!


In [ ]:
def expand_query(query: str, num_expansions: int = 3) -> list:
    """Expand a query to include related terms using Gemini."""
    prompt = f"""
    I need to search a pharmaceutical quality document with this query: "{query}"

    Please help me expand this query by generating {num_expansions} alternative versions that:
    1. Use different but related terminology
    2. Include relevant pharmaceutical/quality terms that might appear in a certificate or SDF
    3. Cover similar concepts but phrased differently

    Format your response as a list of alternative queries only, with no additional text.
    """

    response = llm.complete(prompt)

    expanded_queries = [line.strip() for line in response.text.split('\n') if line.strip()]

    if query not in expanded_queries:
        expanded_queries = [query] + expanded_queries

    return expanded_queries

In [ ]:
expanded = expand_query("What test methods were used for quality control?")
for i, q in enumerate(expanded):
    print(f"{i+1}. {q}")

1. What test methods were used for quality control?
2. <think>
3. Here's a thinking process:
4. 1.  **Analyze User Input:**
5. - **Original Query:** "What test methods were used for quality control?"
6. - **Domain:** Pharmaceutical quality document (likely a Certificate of Analysis, Supplier Declaration Form, Quality Agreement, or similar)
7. - **Requirements:**
8. 1. Use different but related terminology
9. 2. Include relevant pharmaceutical/quality terms that might appear in a certificate or SDF
10. 3. Cover similar concepts but phrased differently
11. 4. Format: List of 3 alternative queries only, no additional text
12. 2.  **Deconstruct Original Query & Identify Key Concepts:**
13. - "test methods" -> analytical procedures, assays, testing protocols, examination techniques, validation methods
14. - "quality control" -> QC, quality assurance, release testing, specification testing, conformity assessment, batch release
15. - Context: Pharmaceutical documents often use terms like "ana

In [ ]:


from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever

def create_query_expansion_engine(index):
    """Create a query engine that uses query expansion."""
    base_retriever = index.as_retriever(similarity_top_k=2)

    fusion_retriever = QueryFusionRetriever(
        retrievers=[base_retriever],
        llm=llm,
        similarity_top_k=2,
        num_queries=3,
        mode="reciprocal_rerank"
    )


    query_engine = RetrieverQueryEngine.from_args(
        retriever=fusion_retriever,
        llm=llm,
        verbose=True
    )

    return query_engine

In [ ]:
expanded_query_engine = create_query_expansion_engine(index)
response = expanded_query_engine.query("What test methods were used for quality control?")
print(response)


<think>
Here's a thinking process:

1.  **Analyze User Query:** The user asks: "What test methods were used for quality control?"
2.  **Scan Context for Keywords:** Look for "Test Method", "quality control", "QC", or specific test names in the provided text.
3.  **Extract Relevant Information:**
   - From page 3:
     - Autoclave - Pump tubing: 121C > 15 min
     - Gamma Irradiation - Inlets: 25.0 - 40.0 kGy
     - Flow Rate Test: Per SOP-QC-042
     - Pressure Integrity: Max 0.5 bar, 5 min hold
     - Visual Inspection: No defects visible
     - Package Integrity: Sealed, no damage
     - Regulatory Conformance: Bio-safety: All polymeric materials in contact with the process fluid comply with United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI.
   - From page 2: Mentions "Product Release Criteria" and "verified to meet predetermined Critical to Quality attributes", but doesn't list specific test methods.
4.  **Synthesize Answer:** List the test methods mentioned

In [ ]:


from llama_index.core import VectorStoreIndex
from llama_index.retrievers.bm25 import BM25Retriever

def create_hybrid_retriever(index, query, top_k=2):

    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    vector_nodes = vector_retriever.retrieve(query)


    nodes = [node for node in index.docstore.docs.values()]
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=top_k
    )
    keyword_nodes = bm25_retriever.retrieve(query)


    all_nodes = []
    all_nodes.extend(vector_nodes)
    all_nodes.extend(keyword_nodes)

    unique_nodes = []
    seen_ids = set()
    for node in all_nodes:
        if node.node_id not in seen_ids:
            unique_nodes.append(node)
            seen_ids.add(node.node_id)


    sorted_nodes = sorted(unique_nodes, key=lambda x: x.score if hasattr(x, 'score') else 0.0, reverse=True)

    top_nodes = sorted_nodes[:top_k]

    return top_nodes

In [ ]:
hybrid_nodes = create_hybrid_retriever(index, "What are the storage conditions for this product?")
for i, node in enumerate(hybrid_nodes):
    print(f"Result {i+1} (Score: {node.score:.4f}):")
    print(node.get_text())
    print("-" * 40)

DEBUG:bm25s:Building index from IDs objects


Result 1 (Score: 0.6012):
Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
Issued by Cytiva Westborough Quality Assurance
This document has been electronically produced and is valid without a signature.
Product:
AKTA ready Low Flow Kit
Lot Number:
18356721
Product Article Number:
28 9301 82
Date of Manufacture:
20240315
Product Description:
Low Flow Kit, AKTA ready
Expiration Date:
20260315
Product Release Criteria
We hereby certify that the defined product has been manufactured to meet its specifications
and have been verified to meet predetermined Critical to Quality attributes.
----------------------------------------
Result 2 (Score: 0.3362):
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section

In [ ]:


from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.schema import NodeWithScore

def rerank_results(nodes, query, top_n=2):

    reranker = SentenceTransformerRerank(
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_n=top_n
    )


    reranked_nodes = reranker.postprocess_nodes(
        nodes,
        query_str=query
    )

    return reranked_nodes

def demonstrate_reranking(index, query, top_k=3):

    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)

    print(f"Query: {query}")
    print("\nOriginal Retrieval Order:")
    for i, node in enumerate(nodes):
        print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")


    reranked_nodes = rerank_results(nodes, query, top_n=2)

    print("\nAfter Reranking:")
    for i, node in enumerate(reranked_nodes):
        print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")


    results = []


    for i, node in enumerate(nodes):
        results.append({
            "Stage": "Original Retrieval",
            "Rank": i + 1,
            "Score": node.score,
            "Content": node.get_text()[:150] + "...",
            "Page": node.metadata.get("page_number", "Unknown")
        })

    for i, node in enumerate(reranked_nodes):
        results.append({
            "Stage": "After Reranking",
            "Rank": i + 1,
            "Score": node.score,
            "Content": node.get_text()[:150] + "...",
            "Page": node.metadata.get("page_number", "Unknown")
        })

    results_df = pd.DataFrame(results)
    display(results_df)

    return results_df

reranking_demo = demonstrate_reranking(index," What test methods were used for quality control?”", top_k=3)

Query:  What test methods were used for quality control?”

Original Retrieval Order:
1. (Score: 0.3757) - Description
Test Method
Result
Autoclave - Pump tubing
121C > 15 min
Conforms
Gamma Irradiation - In...
2. (Score: 0.3266) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...
3. (Score: 0.2745) - Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali...

After Reranking:
1. (Score: -8.3821) - Description
Test Method
Result
Autoclave - Pump tubing
121C > 15 min
Conforms
Gamma Irradiation - In...
2. (Score: -10.4477) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...


,Stage,Rank,Score,Content,Page
0,Original Retrieval,1,-8.382052,Description\nTest Method\nResult\nAutoclave - ...,3
1,Original Retrieval,2,-10.447674,Cytiva\ncytiva.com\nCertificate of Quality\nTh...,3
2,Original Retrieval,3,-10.520130,Certificate of Quality \nThis product is manuf...,2
3,After Reranking,1,-8.382052,Description\nTest Method\nResult\nAutoclave - ...,3
4,After Reranking,2,-10.447674,Cytiva\ncytiva.com\nCertificate of Quality\nTh...,3


In [ ]:

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryBundle

def build_rag_pipeline(index):

    nodes = list(index.docstore.docs.values())


    num_nodes = len(nodes)
    safe_top_k = min(2, max(1, num_nodes))

    print(f"Index contains {num_nodes} nodes, using top_k={safe_top_k}")


    vector_retriever = index.as_retriever(
        similarity_top_k=safe_top_k
    )


    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=safe_top_k
    )

    class HybridRetriever(BaseRetriever):

        def __init__(self, vector_retriever, keyword_retriever, top_k=2):
            self.vector_retriever = vector_retriever
            self.keyword_retriever = keyword_retriever
            self.top_k = top_k
            super().__init__()

        def _retrieve(self, query_bundle, **kwargs):
            vector_nodes = self.vector_retriever.retrieve(query_bundle)
            keyword_nodes = self.keyword_retriever.retrieve(query_bundle)

            all_nodes = list(vector_nodes) + list(keyword_nodes)

            unique_nodes = {}
            for node in all_nodes:
                if node.node_id not in unique_nodes:
                    unique_nodes[node.node_id] = node

            sorted_nodes = sorted(
                unique_nodes.values(),
                key=lambda x: x.score if hasattr(x, 'score') else 0.0,
                reverse=True
            )

            return sorted_nodes[:self.top_k]

    hybrid_retriever = HybridRetriever(
        vector_retriever=vector_retriever,
        keyword_retriever=bm25_retriever,
        top_k=safe_top_k
    )

    if num_nodes > 1:
        reranker = SentenceTransformerRerank(
            model="cross-encoder/ms-marco-MiniLM-L-6-v2",
            top_n=min(2, num_nodes)
        )
        node_postprocessors = [reranker]
    else:
        node_postprocessors = []



    query_engine = RetrieverQueryEngine.from_args(
        retriever=hybrid_retriever,
        llm=llm,
        node_postprocessors=node_postprocessors
    )

    return query_engine

In [ ]:
index = process_and_index_pdf(chunks_overlap)
rag_engine = build_rag_pipeline(index)
response = rag_engine.query("What test methods were used for quality control?")
print('\nFinal Response:\n ---------------------- \n')
print(response)

DEBUG:bm25s:Building index from IDs objects


Vector store created successfully!
Index contains 9 nodes, using top_k=2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Final Response:
 ---------------------- 


<think>
Here's a thinking process:

1.  **Analyze User Query:** The user asks: "What test methods were used for quality control?"
2.  **Scan Context for Keywords:** Look for "Test Method", "quality control", "QC", or specific tests listed in the provided text.
3.  **Extract Relevant Information:**
   - From page 3:
     - Autoclave - Pump tubing: 121C > 15 min
     - Gamma Irradiation - Inlets: 25.0 - 40.0 kGy
     - Flow Rate Test: Per SOP-QC-042
     - Pressure Integrity: Max 0.5 bar, 5 min hold
     - Visual Inspection: No defects visible
     - Package Integrity: Sealed, no damage
   - From page 2: Mentions ISO 9001 certified quality management system, but doesn't list specific test methods.
4.  **Synthesize Answer:** List the test methods mentioned in the context.
   - Autoclave (Pump tubing)
   - Gamma Irradiation (Inlets)
   - Flow Rate Test
   - Pressure Integrity
   - Visual Inspection
   - Package Integrity
   - Also mention the spe